# Comparing commit message keyword sets

This notebook compares a restricted keyword set with the one in the [*When Code Smells Meet ML* replication package](https://figshare.com/articles/dataset/_b_When_Code_Smells_Meet_ML_On_the_Lifecycle_of_ML-specific_Code_Smells_in_ML-enabled_Systems_b_-_Replication_Package/28167065). Both methods use the titles of the same 339 commits from 10 skills.

In [1]:
from pathlib import Path
import pandas as pd

file = Path("../data/gitskills/commit_messages.json")
if not file.exists():
    file = Path("data/gitskills/commit_messages.json")

commits = pd.read_json(file)

## Keyword sets

The replication terms below come from `CodeSmileAnalysis/commit_message_classification.py` in version 3 of the package. Its script looks for word fragments anywhere in the text. The restricted rules further down check prefixes first, then whole words. Both methods are applied only to commit titles here.

In [2]:
keywords = {
    "new feature": {"new", "feature", "feat:", "add", "creat", "introduc", "implement"},
    "bug fixing": {"fix", "repair", "error", "bug", "issue", "exception"},
    "enhancement": {"updat", "chang", "switch", "optimiz", "improv", "edit", "compatibl", "adapt", "updt", "allow", "revise", "modif", "upgrad", "export", "remov", "integrat", "support", "enhanc", "replac", "includ", "expos", "generat", "migrat"},
    "refactoring": {"renam", "restructur", "decoupl", "reorganiz", "rewrit", "rm", "refact", "rework", "ren:", "clean", "polish", "mov", "extract", "reorder", "re-order", "merg"}
}

def classify(text):
    text = text.lower()
    return [label for label, terms in keywords.items() if any(term in text for term in terms)]

commits["subject_labels"] = commits["subject"].apply(classify)

## Results

A commit can have more than one label, so the category counts can add up to more than 339.

In [3]:
import re

restricted_prefixes = {"fix": "bug fixing", "feat": "new feature", "refactor": "refactoring"}
restricted_keywords = {
    "bug fixing": {"fix", "fixed", "fixes", "bug", "bugs", "repair", "correct", "resolve", "resolved"},
    "enhancement": {"improve", "improved", "improvement", "enhance", "enhanced", "optimize", "optimized"},
    "new feature": {"add", "added", "implement", "implemented", "introduce", "introduced", "feature", "features"},
    "refactoring": {"refactor", "refactored", "restructure", "simplify", "cleanup"}
}

def classify_restricted(subject):
    text = subject.lower()
    prefix = text.split(":")[0].split("(")[0].rstrip("!")
    if prefix in restricted_prefixes:
        return [restricted_prefixes[prefix]]

    words = set(re.findall(r"[a-z]+", text))
    return [label for label, terms in restricted_keywords.items() if words & terms]

commits["restricted_labels"] = commits["subject"].apply(classify_restricted)

summary = pd.DataFrame(index=["no label", "one label", "multiple labels", *keywords])

for name, column in [("Restricted keywords", "restricted_labels"), ("Replication", "subject_labels")]:
    labels = commits[column]
    sizes = labels.str.len()
    counts = labels.explode().value_counts()
    summary[name] = [(sizes == 0).sum(), (sizes == 1).sum(), (sizes > 1).sum()] + [counts.get(label, 0) for label in keywords]

summary

,Restricted keywords,Replication
no label,86,60
one label,247,160
multiple labels,6,119
new feature,144,150
bug fixing,64,81
enhancement,8,97
refactoring,43,92


In [4]:
new_matches = commits[(commits["restricted_labels"].str.len() == 0) &
                      (commits["subject_labels"].str.len() > 0)]
restricted_only = commits[(commits["restricted_labels"].str.len() > 0) &
                   (commits["subject_labels"].str.len() == 0)]
print("Newly matched by replication keywords:", len(new_matches))
print("Matched only by restricted keywords:", len(restricted_only))
new_matches[["subject", "subject_labels"]].head(5)

Newly matched by replication keywords: 44
Matched only by restricted keywords: 18


,subject,subject_labels
7,"@ v1.0.8 — approach.js module (State=30), audi...",[new feature]
9,"@ v1.0.8 — English UI dialogs, structured erro...","[new feature, bug fixing]"
17,v1.1.1 — stand position map overlay in editor,[enhancement]
18,"v1.1.2 — StarMap overlay, State=5 approach air...",[refactoring]
19,docs: update repo skill for v1.1.3 changes + A...,[enhancement]


## A false match: `address`

The replication rule searches for `add` anywhere in a title. That also matches the first three letters of `address`. These titles receive a `new feature` label even though they do not say that a feature was added.

In [5]:
examples = commits[commits["subject"].str.contains("address PR", case=False)]
for _, row in examples.iterrows():
    print(row["subject"])
    print("Labels:", row["subject_labels"], "\n")

fix: address PR review feedback
Labels: ['new feature', 'bug fixing'] 

fix: [T703] address PR #151 review — gate conditional pushes to their guard in the drift oracle; sync dispatchSpec across adapter arg-list docs
Labels: ['new feature', 'bug fixing', 'enhancement'] 

fix: [T622] address PR #174 review — restore before fix rounds, path-scope workspace base reads
Labels: ['new feature', 'bug fixing'] 

test: [T706] address PR #186 review — pin reviewer map (AC7) + drive P-SQ control live
Labels: ['new feature'] 



## What this means

The replication keywords match more titles, but also give many more commits multiple labels. They can match unrelated text: `add`, for example, appears inside *address*. These counts show how the rules behave, not how many skill changes truly belong to each category.